# Actor-Critic Policy Evaluation

Evaluate whether the trained policy actually controls the agent meaningfully in imagination.

**Goal**: Demonstrate that the policy can be trained and executed in imagination (the core thesis contribution)

We will:
1. Load trained checkpoints
2. Generate policy-controlled rollouts (learned behavior)
3. Compare against prior rollouts (no learned policy)
4. Visualize action sequences, imagined observations, and predicted values
5. Check if policy learns meaningful behaviors

In [ ]:
import os
os.environ['JAX_PLATFORMS'] = 'cpu'  # Skip CUDA probe (driver/DSO version mismatch after update — remove after reboot)

import json
import yaml
import ruamel.yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
import pickle
import jax
import jax.numpy as jnp
from functools import partial
import importlib

# Setup paths correctly (matching world_model_reconstruction.ipynb)
notebook_dir = Path('/home/maurits-heemskerk/Documents/Uni/Master_Thesis/dreamer_SPOT_implementation/notebooks')
dreamer_dir = Path('/home/maurits-heemskerk/Documents/Uni/Master_Thesis/dreamer_SPOT_implementation/informed-dreamer')

import sys
sys.path.insert(0, str(dreamer_dir))

# Dreamer imports
from dreamerv3 import ninjax as nj
import dreamerv3
import embodied

print(f"✓ JAX version: {jax.__version__}")
print(f"✓ Devices: {jax.devices()}")
print(f"✓ Imports successful")

In [ ]:

import re

results_dir = Path('/home/maurits-heemskerk/Documents/Uni/Master_Thesis/dreamer_results_local_noobs')
available_runs = sorted([d for d in results_dir.iterdir() if d.is_dir()])

print("Available training runs:")
for i, run in enumerate(available_runs):
    ckpt_path   = run / 'checkpoint.ckpt'
    config_path = run / 'config.yaml'
    if ckpt_path.exists() and config_path.exists():
        with open(config_path) as f:
            rc = ruamel.yaml.YAML(typ='safe').load(f)
        enc_mlp = rc.get('encoder', {}).get('mlp_keys', '?')
        print(f"  [{i:2d}] {run.name}  (enc_mlp={enc_mlp!r})")
    else:
        print(f"  [{i:2d}] {run.name} ⚠ (missing checkpoint or config)")

print(f"\n✓ {len(available_runs)} runs found")
print("\n👇 SELECT RUNS AND CHECKPOINT STEPS TO EVALUATE:")

run_indices_to_eval = [78]  # ← modify to select runs
checkpoint_step = 1e7  # ← set to a step value (e.g., 5e6) to load intermediate checkpoint
                        #   if None: loads final checkpoint (default)
                        #   NOTE: only works if intermediate checkpoints were saved during training
print(f"   run_indices_to_eval = {run_indices_to_eval}")
print(f"   checkpoint_step = {checkpoint_step} (None = final checkpoint)")


In [ ]:
def find_checkpoint_path(run_path, checkpoint_step=None):
    """
    Find checkpoint file in run directory, or the closest one to the requested step.
    
    Args:
        run_path: Path to the run directory
        checkpoint_step: Optional training step (e.g., 5e6). If None, uses final checkpoint.
        
    Returns:
        Tuple of (Path to checkpoint file, actual_step_of_checkpoint or None)
        
    NOTE: Most training runs only save the final checkpoint.ckpt (intermediate ones are 
    overwritten). If only the final checkpoint exists, it was trained until the end.
    """
    if checkpoint_step is None:
        # Use final checkpoint
        ckpt_path = run_path / 'checkpoint.ckpt'
        return (ckpt_path if ckpt_path.exists() else None, None)
    
    # Search for all checkpoint files in the directory
    checkpoint_files = []
    
    # Look in root directory: checkpoint.ckpt, checkpoint-*.ckpt, checkpoint_*.ckpt
    for pattern in ['checkpoint.ckpt', 'checkpoint-*.ckpt', 'checkpoint_*.ckpt']:
        checkpoint_files.extend(run_path.glob(pattern))
    
    # Look in checkpoints/ subdirectory: *.ckpt
    checkpoints_dir = run_path / 'checkpoints'
    if checkpoints_dir.exists():
        checkpoint_files.extend(checkpoints_dir.glob('*.ckpt'))
    
    # Remove duplicates
    checkpoint_files = list(set(checkpoint_files))
    
    if not checkpoint_files:
        print(f"⚠ No checkpoints found for step {checkpoint_step:.0e}")
        ckpt_path = run_path / 'checkpoint.ckpt'
        if ckpt_path.exists():
            print(f"  → Using final checkpoint (only checkpoint.ckpt exists)")
            return (ckpt_path, None)
        return (None, None)
    
    # Extract step numbers from filenames
    import re
    step_map = {}  # {step: path}
    
    for ckpt in checkpoint_files:
        fname = ckpt.name
        # Try to extract step number from filename
        # Patterns: checkpoint-1000000.ckpt, checkpoint_1e7.ckpt, 1000000.ckpt
        match = re.search(r'(\d+(?:\.\d+)?e\d+|\d+)', fname)
        if match:
            step_str = match.group(1)
            try:
                if 'e' in step_str:
                    step = float(step_str)
                else:
                    step = int(step_str)
                step_map[step] = ckpt
            except ValueError:
                continue
    
    if not step_map:
        # No named intermediate checkpoints found, use final
        ckpt_path = run_path / 'checkpoint.ckpt'
        if ckpt_path.exists():
            print(f"⚠ Could not parse step numbers from checkpoint filenames")
            print(f"  → Using final checkpoint (only checkpoint.ckpt exists)")
            # Try to extract actual step from metrics.jsonl if available
            metrics_file = run_path / 'metrics.jsonl'
            if metrics_file.exists():
                try:
                    import json
                    with open(metrics_file, 'r') as f:
                        last_line = None
                        for line in f:
                            last_line = line
                        if last_line:
                            last_entry = json.loads(last_line)
                            final_step = last_entry.get('step', None)
                            if final_step:
                                print(f"  → Final checkpoint trained to step: {float(final_step):.0e}")
                                return (ckpt_path, float(final_step))
                except Exception:
                    pass
            return (ckpt_path, None)
        return (None, None)
    
    # Find the closest step to the requested step
    closest_step = min(step_map.keys(), key=lambda s: abs(s - checkpoint_step))
    actual_ckpt_path = step_map[closest_step]
    
    if closest_step != checkpoint_step:
        print(f"  → Requested step {checkpoint_step:.0e}, found closest: {closest_step:.0e}")
    
    return (actual_ckpt_path, closest_step)

In [ ]:
selected_runs = []
run_agents   = {}
run_wms      = {}
run_states   = {}
run_configs  = {}
run_obs_keys = {}
run_ckpt_info = {}  # Track which checkpoint was loaded and its actual step

# Helper function to filter config to known keys
def _filter_to_known(config, defaults):
    """Filter config dict to only include keys that exist in defaults."""
    filtered = {}
    for k, v in config.items():
        if k in defaults:
            if isinstance(v, dict) and isinstance(defaults[k], dict):
                filtered[k] = _filter_to_known(v, defaults[k])
            else:
                filtered[k] = v
    return filtered

# Fixed obs/act space — matched exactly to new checkpoints
# velocity (3,): [vx, vy, wz]  orientation (2,): [cos(yaw), sin(yaw)]  goal (2,): [dx, dy]
# action  (3,): [vx, vy, yaw_rate]  — no gait switching
_SPACE_MAP = {
    'velocity':    embodied.Space(np.float32, (3,)),
    'orientation': embodied.Space(np.float32, (2,)),
    'goal':        embodied.Space(np.float32, (2,)),
    'position':    embodied.Space(np.float32, (2,)),
    'image':       embodied.Space(np.uint8,   (49, 128, 3)),
    'terrain':     embodied.Space(np.float32, (64, 64)),
    'info_terrain':embodied.Space(np.float32, (64, 64)),
}
_ALL_MLP = ['velocity', 'orientation', 'goal', 'position']
_ALL_CNN = ['image', 'terrain', 'info_terrain']

for run_idx in run_indices_to_eval:
    if run_idx >= len(available_runs):
        print(f"⚠ Run index {run_idx} out of range"); continue

    run_path    = available_runs[run_idx]
    config_path = run_path / 'config.yaml'
    
    # Use find_checkpoint_path to support checkpoint_step parameter and find closest
    ckpt_path, actual_step = find_checkpoint_path(run_path, checkpoint_step=checkpoint_step)
    run_ckpt_info[run_idx] = {'path': ckpt_path, 'actual_step': actual_step, 'requested_step': checkpoint_step}

    if not (ckpt_path and ckpt_path.exists() and config_path.exists()):
        print(f"⚠ Run {run_idx} missing checkpoint or config"); continue

    print(f"\nLoading run {run_idx}: {run_path.name}")
    if checkpoint_step is not None:
        if actual_step is not None:
            print(f"  Requested step: {checkpoint_step:.0e}")
            print(f"  Actual checkpoint step: {actual_step:.0e}")
        else:
            print(f"  Checkpoint step: {checkpoint_step:.0e}")
    print("-" * 60)
    try:
        with open(config_path) as f:
            raw_config = ruamel.yaml.YAML(typ='safe').load(f)

        _defaults = embodied.Config(dreamerv3.Agent.configs['defaults'])
        _known_keys = set(_defaults.keys()) - {'env'}
        _filtered = _filter_to_known({k: v for k, v in raw_config.items() if k in _known_keys}, dict(_defaults))

        config = embodied.Config(dreamerv3.Agent.configs['defaults'])
        config = config.update(_filtered)
        config = config.update({'jax.platform': 'cpu', 'jax.prealloc': False})

        _enc_mlp = raw_config['encoder']['mlp_keys']
        _enc_cnn = raw_config['encoder']['cnn_keys']
        mlp_keys = [k for k in _ALL_MLP if _enc_mlp and re.search(_enc_mlp, k)]
        cnn_keys = [k for k in _ALL_CNN if _enc_cnn and re.search(_enc_cnn, k)]
        obs_keys = mlp_keys + cnn_keys
        run_obs_keys[run_idx] = obs_keys

        obs_space = {k: _SPACE_MAP[k] for k in obs_keys if k in _SPACE_MAP}
        obs_space['reward']      = embodied.Space(np.float32)
        obs_space['is_first']    = embodied.Space(bool)
        obs_space['is_last']     = embodied.Space(bool)
        obs_space['is_terminal'] = embodied.Space(bool)
        act_space = {'action': embodied.Space(np.float32, (3,), -1.0, 1.0), 'reset': embodied.Space(bool)}

        print(f"  Obs keys: {obs_keys}")
        print(f"  Act space: action {act_space['action'].shape}  (vx, vy, yaw_rate)")
        print(f"  Loading checkpoint: {ckpt_path.name}")

        with open(ckpt_path, 'rb') as f:
            ckpt_data = pickle.load(f)

        ckpt_state  = ckpt_data['agent']
        agent_state = {}
        wm_keys = ac_keys = 0
        for k, v in ckpt_state.items():
            if k.startswith('agent/'):
                new_k = k[6:]
                agent_state[new_k] = v
                if 'wm/' in k:           wm_keys += 1
                if 'task_behavior' in k: ac_keys  += 1

        step  = embodied.Counter()
        agent = dreamerv3.Agent(obs_space, act_space, step, config)
        jax.config.update('jax_transfer_guard', 'allow')

        from dreamerv3.agent import WorldModel
        wm = WorldModel(obs_space, act_space, config, name='wm')

        run_agents[run_idx]  = (agent, agent_state)
        run_wms[run_idx]     = wm
        run_states[run_idx]  = agent_state
        run_configs[run_idx] = config
        selected_runs.append(run_idx)

        print(f"  ✓ WM parameters: {wm_keys}")
        print(f"  ✓ Actor-Critic parameters: {ac_keys}")
        print(f"  ✓ Task behavior: {config.task_behavior}")

    except Exception as e:
        print(f"  ✗ Error: {e}")
        import traceback; traceback.print_exc()

print(f"\n{'='*60}")
print(f"✓ Loaded {len(selected_runs)} agents successfully")

In [ ]:

# Quick debug: check what obs_space was used during training
print("\n" + "="*80)
print("CHECKPOINT TRAINING CONFIG INSPECTION")
print("="*80)

for run_idx in run_indices_to_eval[:1]:  # Just first run for inspection
    run_path = available_runs[run_idx]
    config_path = run_path / 'config.yaml'
    
    with open(config_path) as f:
        raw_config = ruamel.yaml.YAML(typ='safe').load(f)
    
    print(f"\nRun {run_idx}: {run_path.name}")
    print(f"Encoder mlp_keys: {raw_config['encoder'].get('mlp_keys', 'NOT SET')}")
    print(f"Encoder cnn_keys: {raw_config['encoder'].get('cnn_keys', 'NOT SET')}")
    
    # The decoder heads are determined by obs_space, which is determined by encoder config
    # If is_terminal was in training data, it would have been in obs_space
    # Let's check what the full env obs_space should be based on this config
    _enc_mlp = raw_config['encoder']['mlp_keys']
    _enc_cnn = raw_config['encoder']['cnn_keys']
    mlp_keys = [k for k in _ALL_MLP if _enc_mlp and re.search(_enc_mlp, k)]
    cnn_keys = [k for k in _ALL_CNN if _enc_cnn and re.search(_enc_cnn, k)]
    obs_keys_during_training = mlp_keys + cnn_keys
    
    print(f"\nObs keys that were encoded: {obs_keys_during_training}")
    print(f"Control signals added (always in obs_space): reward, is_first, is_last, is_terminal (maybe)")
    print(f"\n→ If is_terminal WAS in decoder heads during training, the checkpoint should have:")
    print(f"  weights/parameters for a 'is_terminal' output head in the decoder")
    print(f"  But if it WASN'T, manually adding it to obs_space creates a fresh head (random init)")


## Policy Rollout Functions

Generate trajectories under:
1. **Prior**: No learned policy (sample random actions from prior)
2. **Policy**: Learned actor-critic policy controls action selection

In [ ]:

def rollout_with_prior(wm, agent_state, obs_batch, action_batch, obs_keys,
                       horizon=70, seed=0, start_step=30):
    """
    Random-action prior rollout through the world model.
    """
    _state_keys = [k for k in obs_keys if k in ['velocity', 'orientation', 'goal', 'position']]

    def prior_imagination():
        first_obs    = {k: v[:, start_step:start_step + 1] for k, v in obs_batch.items()}
        first_action = jnp.zeros((obs_batch['is_first'].shape[0], 1, 3), dtype=jnp.float32)
        embed_first  = wm.encoder(first_obs)
        post_first, _ = wm.rssm.observe(embed_first, first_action, first_obs['is_first'])
        start_state  = {k: v[:, 0] for k, v in post_first.items()}

        batch_size     = obs_batch['is_first'].shape[0]
        random_actions = jnp.tanh(jax.random.normal(nj.rng(), shape=(batch_size, horizon, 3)))
        priors         = wm.rssm.imagine(random_actions, start_state)
        recons_dists   = wm.heads['decoder'](priors)
        extra = {k: recons_dists[k].mean() for k in _state_keys if k in recons_dists}
        return extra, jnp.transpose(random_actions, (1, 0, 2))

    rng_key = jax.random.PRNGKey(seed)
    (extra, actions), _ = nj.pure(prior_imagination)(agent_state, rng_key)

    rollout = {'mode': 'prior', 'actions': actions}
    for k, v in extra.items():
        rollout[k] = np.array(v).squeeze(0)
    return rollout, {}

print("✓ Prior rollout ready (random actions, 3-dim act space)")

In [ ]:

def rollout_with_policy(agent, wm, agent_state, obs_batch, action_batch, obs_keys,
                        horizon=70, seed=1, start_step=10):
    """
    Policy-driven rollout through the world model.
    Action space: (3,) = [vx, vy, yaw_rate] — no gait switching.
    """
    eval_state = dict(agent_state)
    for k, v in agent_state.items():
        if k.startswith('task_behavior/'):
            eval_state['agent/' + k] = v

    _state_keys = [k for k in obs_keys if k in ['velocity', 'orientation', 'goal', 'position']]

    def policy_imagination():
        first_obs    = {k: v[:, start_step:start_step + 1] for k, v in obs_batch.items()}
        first_action = jnp.zeros((obs_batch['is_first'].shape[0], 1, 3), dtype=jnp.float32)
        embed_first  = wm.encoder(first_obs)
        post_first, _ = wm.rssm.observe(embed_first, first_action, first_obs['is_first'])
        latent = {k: v[:, 0] for k, v in post_first.items()}

        actor      = agent.agent.task_behavior.ac.actor
        critic_net = agent.agent.task_behavior.ac.critics['extr'].net

        latent_list, action_list, value_list, entropy_list = [], [], [], []

        for _ in range(horizon):
            action_dist = actor(latent)
            action  = action_dist.mode()
            entropy = action_dist.entropy()
            value   = critic_net(latent).mean()

            latent_list.append(latent)
            action_list.append(action)
            value_list.append(value)
            entropy_list.append(entropy)

            next_latent = wm.rssm.imagine(jnp.expand_dims(action, axis=1), latent)
            latent = {k: v[:, 0] for k, v in next_latent.items()}

        reward_list = [wm.heads['reward'](l).mean() for l in latent_list]
        cont_list   = [wm.heads['cont'](l).mean()   for l in latent_list]

        latents_BH   = {k: jnp.stack([l[k] for l in latent_list], axis=1) for k in latent_list[0]}
        recons_dists = wm.heads['decoder'](latents_BH)
        extra = {k: recons_dists[k].mean() for k in _state_keys if k in recons_dists}

        # cont head: P(episode continues). 1=continues, 0=terminated.
        # is_terminal = 1 - cont. This is what drives value bootstrapping cutoff.
        extra['cont'] = jnp.stack(cont_list)  # (H, B)

        return (
            jnp.stack(action_list),    # (H, B, 3)
            jnp.stack(value_list),     # (H, B)
            jnp.stack(reward_list),    # (H, B)
            jnp.stack(entropy_list),   # (H, B)
            extra,
        )

    rng_key = jax.random.PRNGKey(seed)
    (actions, values, rewards, entropies, extra), _ = nj.pure(policy_imagination)(eval_state, rng_key)

    # cont is (H, B) — extract before the squeeze loop (same shape as values/rewards)
    cont = extra.pop('cont', None)

    rollout = {
        'mode':      'policy',
        'actions':   actions,
        'values':    values,
        'rewards':   rewards,
        'entropies': entropies,
    }
    if cont is not None:
        rollout['cont'] = cont  # (H, B) — access as rollout['cont'][:, 0]

    for k, v in extra.items():
        rollout[k] = np.array(v).squeeze(0)

    return rollout, agent_state

print("✓ Policy rollout ready (actor-critic, 3-dim act space: vx, vy, yaw_rate)")
print("✓ cont head (episode termination) extracted alongside values/rewards")

## Load Data Sample for Initialization

We need example observations from the dataset to initialize the world model's posterior.

In [ ]:

import h5py

# ── Reward function parameters (keep in sync with compute_rewards_batch_noobs.py) ──
REWARD_PARAMS = {
    'distance_scale':       1.0,   # multiplier on per-step distance change
    'goal_reach_threshold': 0.5,   # metres — radius of goal zone for bonus
    'goal_bonus':           0.5,   # per-step reward while inside goal zone
    'orientation_scale':    0.1,   # weight of heading-to-goal penalty
}


def compute_gt_reward(states, goal_xy=None):
    """
    Compute GT rewards from raw state data using the current reward function.
    r(t) = progress + goal_bonus * at_goal + orientation_penalty
      progress          = -distance_scale * (dist(t) - dist(t-1))
      goal_bonus        = goal_bonus per step while dist < goal_reach_threshold
      orientation_penalty = -orientation_scale * (1 - cos(heading_error)) / 2

    states:  (T, 7) [x, y, z, qx, qy, qz, qw]
    goal_xy: (2,)   defaults to final position
    """
    if goal_xy is None:
        goal_xy = states[-1, :2].astype(np.float32)

    # Progress reward
    dist_to_goal = np.linalg.norm(states[:, :2] - goal_xy, axis=1)
    delta_dist   = np.diff(dist_to_goal, prepend=dist_to_goal[0])   # r(0) = 0
    progress     = -REWARD_PARAMS['distance_scale'] * delta_dist

    # Goal bonus
    at_goal = (dist_to_goal < REWARD_PARAMS['goal_reach_threshold']).astype(np.float32)

    # Orientation-to-goal penalty
    qx_s, qy_s, qz_s, qw_s = states[:, 3], states[:, 4], states[:, 5], states[:, 6]
    yaw_s   = np.arctan2(2.0 * (qw_s * qz_s + qx_s * qy_s),
                         1.0 - 2.0 * (qy_s ** 2 + qz_s ** 2))
    heading     = np.stack([np.cos(yaw_s), np.sin(yaw_s)], axis=1)
    goal_dir    = goal_xy - states[:, :2]
    goal_dist2d = np.linalg.norm(goal_dir, axis=1, keepdims=True)
    goal_dir_norm = goal_dir / np.maximum(goal_dist2d, 1e-6)
    cos_angle   = np.sum(heading * goal_dir_norm, axis=1)
    orientation_penalty = -REWARD_PARAMS['orientation_scale'] * (1.0 - cos_angle) / 2.0

    rewards = progress + REWARD_PARAMS['goal_bonus'] * at_goal + orientation_penalty
    return rewards.astype(np.float32)


data_dir = Path('/home/maurits-heemskerk/Documents/Uni/Master_Thesis/dreamer_SPOT_implementation/informed-dreamer/processed_data_NoObs_with_rewards')
h5_files = sorted(data_dir.glob('**/*.h5'))

if not h5_files:
    print("⚠ No h5 files found.")
    _raw_data = None
else:
    episode_file = h5_files[0]
    print(f"Loading: {episode_file.name}")

    with h5py.File(episode_file, 'r') as f:
        velocities_raw = f['observations/velocities'][:].astype(np.float32)  # (T, 6)
        states         = f['observations/state'][:].astype(np.float32)        # (T, 7)
        actions_data   = f['actions'][:].astype(np.float32)

    seq_length = len(velocities_raw)

    # ── Exact feature extraction — matched to new checkpoints ─────────────────
    # velocity (3,): [vx, vy, wz]  — drop vz (idx 2), wx/wy pitch+roll (idx 3,4)
    velocities = velocities_raw[:, [0, 1, 5]].astype(np.float32)

    # orientation (2,): [cos(yaw), sin(yaw)] — absolute, not episode-relative
    qx, qy, qz, qw = states[:, 3], states[:, 4], states[:, 5], states[:, 6]
    yaw = np.arctan2(2.0 * (qw * qz + qx * qy), 1.0 - 2.0 * (qy * qy + qz * qz))
    orientations = np.stack([np.cos(yaw), np.sin(yaw)], axis=-1).astype(np.float32)

    # position (2,): zero-centred — reference only, not fed to encoder
    xy_raw    = states[:, :2]
    positions = (xy_raw - xy_raw[0]).astype(np.float32)

    # goal (2,): world-frame displacement to episode-final position
    dx = xy_raw[-1, 0] - xy_raw[:, 0]
    dy = xy_raw[-1, 1] - xy_raw[:, 1]
    goal_relative = np.stack([dx, dy], axis=-1).astype(np.float32)

    # GT reward: computed from current reward function (not read from h5)
    rewards_gt = compute_gt_reward(states)

    _raw_data = {
        'velocity':    velocities,
        'orientation': orientations,
        'goal':        goal_relative,
        'position':    positions,
        'reward':      rewards_gt,
    }
    # Slice actions to 3-dim (vx, vy, yaw_rate) in case H5 stores 4-dim
    actions_batch_real = np.expand_dims(actions_data[:seq_length, :3], 0)

    print(f"✓ Episode length: {seq_length}")
    print(f"✓ Available observation arrays: {list(_raw_data.keys())}")
    print(f"✓ Velocity shape: {velocities.shape}  (vx, vy, wz)")
    print(f"✓ Orientation shape: {orientations.shape}  (cos, sin)")
    print(f"✓ Action shape: {actions_batch_real.shape}  (vx, vy, yaw_rate)")
    print(f"✓ GT reward (computed): min={rewards_gt.min():.3f}  max={rewards_gt.max():.3f}  sum={rewards_gt.sum():.3f}")


def make_obs_batch(obs_keys, seq_len=None):
    """Build obs_batch for the given obs_keys from the loaded episode."""
    n = seq_len or seq_length
    batch = {k: np.expand_dims(_raw_data[k][:n], 0)
             for k in obs_keys if k in _raw_data}
    batch['is_first']    = np.zeros((1, n), dtype=bool)
    batch['is_last']     = np.zeros((1, n), dtype=bool)
    batch['is_terminal'] = np.zeros((1, n), dtype=bool)
    batch['reward']      = np.expand_dims(_raw_data['reward'][:n], 0)
    batch['is_first'][0, 0]  = True
    batch['is_last'][0, -1]  = True
    return batch

print(f"\n✓ make_obs_batch() ready")
print(f"✓ compute_gt_reward() defined  (progress + goal_bonus + orientation_penalty)")


## Visualization Setup

Tools for visualizing rollouts

In [ ]:

import os
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image as IPyImage

# 3-dim action space: vx, vy, yaw_rate — no gait
ACTION_LABELS = ['vel_x (fwd)', 'vel_y (lat)', 'vel_yaw']


def _orientation_to_yaw(ori):
    ori = np.asarray(ori)
    if ori.ndim != 2:
        raise ValueError(f"Expected orientation shape (H, D), got {ori.shape}")
    if ori.shape[1] == 2:
        return np.arctan2(ori[:, 1], ori[:, 0])
    if ori.shape[1] == 4:
        qx, qy, qz, qw = ori[:, 0], ori[:, 1], ori[:, 2], ori[:, 3]
        return np.arctan2(
            2.0 * (qw * qz + qx * qy),
            1.0 - 2.0 * (qy ** 2 + qz ** 2),
        )
    raise ValueError(f"Unsupported orientation dim {ori.shape[1]}; expected 2 or 4")


def plot_action_sequences(rollout_policy, run_label, max_steps=30):
    fig, ax = plt.subplots(figsize=(14, 4))
    policy_actions = np.array(rollout_policy['actions'][:max_steps, 0, :])
    colors = ['tab:blue', 'tab:orange', 'tab:green']
    for i, (label, color) in enumerate(zip(ACTION_LABELS, colors)):
        ax.plot(policy_actions[:, i], label=label, color=color, linewidth=1.5, alpha=0.85)
    ax.set_title(f"{run_label} — Policy Actions (Learned)", fontsize=11, fontweight='bold')
    ax.set_ylabel('Action value'); ax.set_xlabel('Imagination step')
    ax.set_ylim(-1.2, 1.2); ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3); ax.axhline(y=0, color='k', linestyle='--', alpha=0.2)
    plt.tight_layout()
    return fig


def plot_value_predictions(rollout_policy, run_label, max_steps=30):
    fig, ax = plt.subplots(figsize=(12, 4))
    values  = np.array(rollout_policy['values'][:max_steps, 0])
    rewards = np.array(rollout_policy['rewards'][:max_steps, 0])
    ax.plot(values,  label='Critic value estimate', linewidth=2, marker='o', markersize=4)
    ax.plot(rewards, label='Predicted reward (decoded)',        linewidth=2, marker='s', markersize=4, alpha=0.7, linestyle=':')
    ax.set_title(f"{run_label} — Value & Reward During Policy Rollout", fontsize=11, fontweight='bold')
    ax.set_xlabel('Imagination step'); ax.set_ylabel('Value / Reward')
    ax.legend(loc='best', fontsize=10); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig


def create_policy_state_gif(rollout_policy, run_label, obs_batch_ref=None,
                             output_path='/tmp/policy_dream.gif', fps=3, dt=0.5):
    has_pos  = 'position'    in rollout_policy
    has_vel  = 'velocity'    in rollout_policy
    has_ori  = 'orientation' in rollout_policy
    has_goal = 'goal'        in rollout_policy

    actions_arr  = np.array(rollout_policy['actions'])   # (H, B, 3)
    values_arr   = np.array(rollout_policy['values'])    # (H, B)
    rewards_arr  = np.array(rollout_policy['rewards'])   # (H, B)
    horizon      = actions_arr.shape[0]
    print(f"  Creating GIF: {horizon} frames at {fps} fps...")

    yaw = _orientation_to_yaw(rollout_policy['orientation']) if has_ori else np.zeros(horizon)

    if has_pos:
        position = rollout_policy['position']
        traj_x, traj_y = position[:, 0], position[:, 1]
        dr_x, dr_y = np.zeros(horizon), np.zeros(horizon)
        dr_x[0], dr_y[0] = traj_x[0], traj_y[0]
        for _t in range(horizon - 1):
            vx, vy = float(actions_arr[_t, 0, 0]), float(actions_arr[_t, 0, 1])
            cy, sy = np.cos(yaw[_t]), np.sin(yaw[_t])
            dr_x[_t+1] = dr_x[_t] + (vx*cy - vy*sy) * dt
            dr_y[_t+1] = dr_y[_t] + (vx*sy + vy*cy) * dt

    yaw = _orientation_to_yaw(rollout_policy['orientation']) if has_ori else np.zeros(horizon)
    heading_len = 0.75
    heading_line = None
    orient_line = orient_tip = None

    fig, axes_g = plt.subplots(2, 3, figsize=(20, 10))
    fig.suptitle(f"{run_label} — Policy Dream Recording", fontsize=12, fontweight='bold')
    ax_traj, ax_vel, ax_goal, ax_orient, ax_info, ax_blank = axes_g.flatten()

    # ── Trajectory ───────────────────────────────────────────────────────────
    if has_pos:
        if obs_batch_ref is not None and 'position' in obs_batch_ref:
            gt_pos = obs_batch_ref['position'][0] if obs_batch_ref['position'].ndim == 3 \
                     else obs_batch_ref['position']
            ax_traj.plot(gt_pos[:, 0], gt_pos[:, 1], 'b--', linewidth=1.5, alpha=0.5, label='Real (init)')
            all_x = np.concatenate([traj_x, dr_x, gt_pos[:, 0]])
            all_y = np.concatenate([traj_y, dr_y, gt_pos[:, 1]])
        else:
            all_x = np.concatenate([traj_x, dr_x])
            all_y = np.concatenate([traj_y, dr_y])
        pad = max(0.5, (all_x.max()-all_x.min())*0.15 + 0.5)
        ax_traj.set_xlim(all_x.min()-pad, all_x.max()+pad)
        ax_traj.set_ylim(all_y.min()-pad, all_y.max()+pad)
        ax_traj.plot(traj_x, traj_y, 'r:', linewidth=1, alpha=0.15)
        ax_traj.plot(dr_x,   dr_y,   color='gold', linewidth=1, alpha=0.15, linestyle='--')
        line_traj,  = ax_traj.plot([], [], 'r:', linewidth=2, label='Decoded pos (dotted)')
        point_traj, = ax_traj.plot([], [], 'r*', markersize=10)
        line_dr,    = ax_traj.plot([], [], color='gold', linewidth=2, linestyle='--', label='Cmd dead-reckoned')
        point_dr,   = ax_traj.plot([], [], '*', color='gold', markersize=10)
        if has_ori:
            heading_len = max(0.4, pad * 0.25)
            heading_line, = ax_traj.plot([], [], color='deepskyblue', linewidth=2, label='Heading')
        ax_traj.set_aspect('equal', adjustable='datalim'); ax_traj.legend(fontsize=7)
    else:
        ax_traj.text(0.5, 0.5, 'Position not used', ha='center', va='center',
                     transform=ax_traj.transAxes)
        line_traj = point_traj = line_dr = point_dr = None
        traj_x = traj_y = None
    ax_traj.set_title('Position  (red=decoded  gold=cmd)', fontsize=10, fontweight='bold')
    ax_traj.set_xlabel('X (m)'); ax_traj.set_ylabel('Y (m)'); ax_traj.grid(True, alpha=0.3)

    # ── Velocity time series ─────────────────────────────────────────────────
    vel_lines_dec = []
    vel_lines_cmd = []
    for dim, c in enumerate(['tab:blue', 'tab:orange']):
        l_d, = ax_vel.plot([], [], ':',  color=c, linewidth=2, label=f'Decoded vel[{dim}] ')
        l_c, = ax_vel.plot([], [], '--', color=c, linewidth=1.5, label=f'Cmd vel[{dim}]', alpha=0.7)
        vel_lines_dec.append(l_d); vel_lines_cmd.append(l_c)
    ax_vel.set_xlim(0, horizon-1); ax_vel.set_ylim(-1.3, 1.3)
    ax_vel.set_title('Velocity x/y  (decoded vs commanded)', fontsize=10, fontweight='bold')
    ax_vel.set_xlabel('Step'); ax_vel.set_ylabel('m/s  /  cmd-unit')
    ax_vel.legend(fontsize=7, loc='upper right'); ax_vel.grid(True, alpha=0.3)

    # ── Goal (ego-centric) ───────────────────────────────────────────────────
    if has_goal:
        goal_arr = rollout_policy['goal']
        g_range  = max(1.0, float(np.abs(goal_arr).max()) + 0.5)
        ax_goal.scatter(0, 0, color='blue', s=200, marker='o', zorder=5,
                        edgecolors='black', linewidths=2, label='Robot (0,0)')
        point_goal, = ax_goal.plot([], [], '*', color='gold', markersize=14,
                                   label='Goal (decoded)', zorder=7,
                                   markeredgecolor='black', markeredgewidth=0.5)
        ax_goal.set_xlim(-g_range, g_range); ax_goal.set_ylim(-g_range, g_range)
        ax_goal.set_aspect('equal', adjustable='datalim'); ax_goal.legend(fontsize=7)
    else:
        ax_goal.text(0.5, 0.5, 'Goal not used', ha='center', va='center',
                     transform=ax_goal.transAxes)
        point_goal = None
    ax_goal.set_title('Goal (Ego-Centric)', fontsize=10, fontweight='bold')
    ax_goal.set_xlabel('X (m)'); ax_goal.set_ylabel('Y (m)'); ax_goal.grid(True, alpha=0.3)

    # ── Orientation compass ─────────────────────────────────────────────────
    if has_ori:
        ax_orient.set_aspect('equal', adjustable='box')
        ax_orient.add_patch(plt.Circle((0, 0), 1.0, fill=False, color='lightgray',
                                       linestyle='--', linewidth=1))
        ax_orient.axhline(0, color='k', linewidth=0.5, alpha=0.3)
        ax_orient.axvline(0, color='k', linewidth=0.5, alpha=0.3)
        orient_line, = ax_orient.plot([], [], color='deepskyblue', linewidth=2, label='Heading')
        orient_tip, = ax_orient.plot([], [], 'o', color='deepskyblue', markersize=6)
        ax_orient.set_xlim(-1.2, 1.2); ax_orient.set_ylim(-1.2, 1.2)
        ax_orient.legend(fontsize=7, loc='upper right')
    else:
        ax_orient.text(0.5, 0.5, 'Orientation not used', ha='center', va='center',
                       transform=ax_orient.transAxes)
        orient_line = orient_tip = None
    ax_orient.set_title('Orientation (decoded heading)', fontsize=10, fontweight='bold')
    ax_orient.set_xlabel('cos(yaw)'); ax_orient.set_ylabel('sin(yaw)'); ax_orient.grid(True, alpha=0.3)

    ax_blank.axis('off')

    # ── Info panel ───────────────────────────────────────────────────────────
    ax_info.axis('off')
    step_text = ax_info.text(0.5, 0.93, '', ha='center', fontsize=12,
                             fontweight='bold', transform=ax_info.transAxes)
    info_text = ax_info.text(0.05, 0.75, '', ha='left', fontsize=9,
                             family='monospace', transform=ax_info.transAxes)

    def update_frame(t):
        act_t = actions_arr[t, 0, :]   # (3,)

        if line_traj is not None:
            line_traj.set_data(traj_x[:t+1], traj_y[:t+1])
            point_traj.set_data([traj_x[t]], [traj_y[t]])
            line_dr.set_data(dr_x[:t+1], dr_y[:t+1])
            point_dr.set_data([dr_x[t]], [dr_y[t]])
            if heading_line is not None:
                hx = traj_x[t] + heading_len * np.cos(yaw[t])
                hy = traj_y[t] + heading_len * np.sin(yaw[t])
                heading_line.set_data([traj_x[t], hx], [traj_y[t], hy])

        for dim in range(2):
            vel_lines_dec[dim].set_data(range(t+1),
                [rollout_policy['velocity'][s, dim] if has_vel else 0.0 for s in range(t+1)])
            vel_lines_cmd[dim].set_data(range(t+1),
                [float(actions_arr[s, 0, dim]) for s in range(t+1)])

        if point_goal is not None:
            point_goal.set_data([goal_arr[t, 0]], [goal_arr[t, 1]])
        if orient_line is not None:
            orient_line.set_data([0, np.cos(yaw[t])], [0, np.sin(yaw[t])])
            orient_tip.set_data([np.cos(yaw[t])], [np.sin(yaw[t])])

        step_text.set_text(f'Step {t} / {horizon-1}')
        pos_str  = f'Pos:  ({traj_x[t]:.2f}, {traj_y[t]:.2f})' if traj_x is not None else 'Pos: N/A'
        goal_str = f'Goal: ({goal_arr[t,0]:.2f}, {goal_arr[t,1]:.2f})' if has_goal else 'Goal: N/A'
        vel_str  = (f'Vel:  ({rollout_policy["velocity"][t,0]:.2f}, {rollout_policy["velocity"][t,1]:.2f})'
                    if has_vel else 'Vel: N/A')
        yaw_str  = f'Yaw:  {np.degrees(yaw[t]):+.1f} deg' if has_ori else 'Yaw: N/A'
        info_text.set_text(
            f'{pos_str}\n{vel_str}\n'
            f'Cmd:  vx={act_t[0]:+.2f}  vy={act_t[1]:+.2f}  yaw={act_t[2]:+.2f}\n'
            f'{yaw_str}\n'
            f'{goal_str}\n'
            f'Value:  {values_arr[t,0]:.3f}\n'
            f'Reward: {rewards_arr[t,0]:.3f}'
        )
        return [step_text, info_text]

    anim = FuncAnimation(fig, update_frame, frames=horizon,
                         interval=int(1000/fps), blit=False, repeat=True)
    anim.save(output_path, writer=PillowWriter(fps=fps))
    sz = os.path.getsize(output_path) / (1024*1024)
    print(f"  ✓ GIF saved: {output_path} ({sz:.1f} MB, {horizon/fps:.1f}s at {fps} fps)")
    plt.close(fig)
    from IPython.display import display as ipy_display
    ipy_display(IPyImage(filename=output_path))


print("✓ Visualization functions defined  (3-dim action space)")


In [ ]:
def plot_latent_drift(drift_dict, run_label):
    fig, axes = plt.subplots(1, 2, figsize=(14, 3))
    for ax, (key, drift) in zip(axes, drift_dict.items()):
        ax.plot(drift, linewidth=2)
        ax.set_title(f"{run_label} — Latent Drift ({key})", fontsize=10, fontweight='bold')
        ax.set_xlabel("Timestep"); ax.set_ylabel("L2 Distance"); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig


def plot_policy_entropy(rollout_policy, run_label):
    entropies = np.array(rollout_policy.get('entropies', []))
    if entropies.size == 0:
        print("  ⚠ No entropy data in rollout."); return None
    ent = entropies[:, 0]
    fig = plt.figure(figsize=(10, 3))
    plt.plot(ent, linewidth=2, color='darkorange')
    plt.axhline(float(np.mean(ent)), linestyle='--', color='gray', alpha=0.6,
                label=f'mean = {float(np.mean(ent)):.3f}')
    plt.title(f"{run_label} — Actor Policy Entropy", fontsize=11, fontweight='bold')
    plt.xlabel("Imagination step"); plt.ylabel("Entropy (nats)")
    plt.legend(fontsize=9); plt.grid(True, alpha=0.3); plt.tight_layout()
    return fig


def plot_reconstruction_error(errors, run_label):
    keys = list(errors.keys())
    if not keys: return None
    fig, axes = plt.subplots(1, len(keys), figsize=(4*len(keys), 3))
    if len(keys) == 1: axes = [axes]
    for ax, key in zip(axes, keys):
        ax.plot(errors[key], linewidth=2)
        ax.set_title(f"{key} MSE", fontsize=10, fontweight='bold')
        ax.set_xlabel("Timestep"); ax.set_ylabel("MSE"); ax.grid(True, alpha=0.3)
    fig.suptitle(f"{run_label} — Decoder Reconstruction Error (real obs)",
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    return fig


print("✓ Diagnostics functions defined  (3-dim action space, no gait metric)")

In [ ]:
def plot_cont_prediction(rollout_policy, run_label, config=None):
    """
    Visualize the cont head prediction over the imagination horizon.
    cont = P(episode continues). is_terminal = 1 - cont.
    This is the actual mechanism that gates value bootstrapping — when cont→0,
    the TD discount zeroes out, which explains value spikes at the goal.
    """
    if 'cont' not in rollout_policy:
        print(f"  ⚠ cont not in rollout")
        return None

    cont_pred = np.array(rollout_policy['cont'])
    if cont_pred.ndim == 2:
        cont_pred = cont_pred[:, 0]  # take batch dim 0
    terminal_pred = 1.0 - cont_pred  # is_terminal probability
    H = len(cont_pred)
    steps = np.arange(H)

    # Use actual config horizon — critical for correct gamma display
    horizon_val = config.horizon if config is not None else 333
    gamma = 1.0 - 1.0 / horizon_val

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    fig.suptitle(f"{run_label} — Cont Head: Episode Continuation Prediction  (γ=1-1/{horizon_val:.0f}={gamma:.4f})", fontsize=12, fontweight='bold')

    # Panel 1: cont probability (should stay near 1 while episode ongoing, drop at goal/end)
    ax1.plot(steps, cont_pred, 'g-o', linewidth=2, markersize=5, label='P(cont) = P(episode continues)')
    ax1.plot(steps, terminal_pred, 'r--o', linewidth=2, markersize=5, alpha=0.7, label='P(terminal) = 1 - cont')
    ax1.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5, label='0.5 threshold')
    ax1.fill_between(steps, 0, terminal_pred, alpha=0.2, color='red', label='Termination probability area')
    ax1.set_ylabel('Probability', fontsize=11)
    ax1.set_ylim(-0.05, 1.05)
    ax1.legend(fontsize=9, loc='best')
    ax1.grid(True, alpha=0.3)
    ax1.set_title('Continuation vs Termination Probability  (drives TD bootstrap weighting)', fontsize=10)

    # Panel 2: effective discount factor per step (cont * gamma)
    eff_discount = cont_pred * gamma
    ax2.plot(steps, eff_discount, 'purple', linewidth=2, label=f'Effective discount (cont × γ={gamma:.4f})')
    ax2.axhline(y=gamma, color='gray', linestyle='--', alpha=0.5, label=f'Full discount γ={gamma:.4f}')
    ax2.fill_between(steps, eff_discount, gamma, alpha=0.3, color='red',
                     label='Discount reduction from termination')
    ax2.set_ylabel('Effective discount', fontsize=11)
    ax2.set_xlabel('Imagination step', fontsize=11)
    ax2.set_ylim(0, 1.0)
    ax2.legend(fontsize=9, loc='best')
    ax2.grid(True, alpha=0.3)
    ax2.set_title('Effective TD Discount  (where cont drops → value bootstrap cut off)', fontsize=10)

    # Mark where cont drops below 0.5
    terminal_steps = np.where(cont_pred < 0.5)[0]
    if len(terminal_steps) > 0:
        first_idx = terminal_steps[0]
        ax1.axvline(x=first_idx, color='darkred', linestyle='--', linewidth=2, alpha=0.7)
        ax2.axvline(x=first_idx, color='darkred', linestyle='--', linewidth=2, alpha=0.7)
        info_str = f"cont < 0.5 (predicted terminal) first at step {first_idx}  (cont={cont_pred[first_idx]:.3f})"
    else:
        info_str = f"No termination predicted in {H}-step horizon  (min cont={np.min(cont_pred):.3f})"

    ax1.text(0.02, 0.05, info_str, transform=ax1.transAxes, fontsize=10,
             verticalalignment='bottom', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    plt.tight_layout()
    return fig


print("✓ plot_cont_prediction defined  (gamma now reads from config.horizon)")


## Command–State Alignment

Do commanded actions produce the expected motion in the decoded state?

Five panels:
1. **Trajectory**: decoded position (red) vs. dead-reckoned position from vel_x/vel_y commands (blue dashed). Dead-reckoning uses decoded yaw to rotate body-frame commands into world frame with a fixed `dt` (step interval). If the world model is consistent, both traces should point in the same direction even if they diverge over time due to delay.
2. **Speed alignment**: decoded forward speed (red, `velocity[:, 0]`) vs. commanded forward speed (blue, `actions[:, 0]`). Expect positive correlation with a lag.
3. **Lateral speed alignment**: same for lateral axis.
4. **Orientation**: decoded yaw over time, compared against the reference orientation when available.
5. **Goal distance**: decoded ego-centric goal distance over time. Should decrease if the policy is navigating toward the goal.


In [ ]:
def plot_command_state_alignment(rollout_policy, run_label, dt=0.5):
    """
    Two-panel figure:
    (a) Decoded goal dx and dy (imagined)
    (b) Decoded reward from world model (imagined)
    """
    has_goal   = 'goal'    in rollout_policy
    has_reward = 'rewards' in rollout_policy

    actions = np.array(rollout_policy['actions'])[:, 0, :]
    H       = len(actions)
    steps   = np.arange(H)

    colors = list(plt.cm.tab10.colors)

    fig, axes = plt.subplots(2, 1, figsize=(10, 4 * 2), squeeze=False)

    # ── (a) Decoded goal dx and dy ────────────────────────────────────────────
    ax = axes[0, 0]
    if has_goal:
        goal = np.array(rollout_policy['goal'])
        goal_labels = ['dx', 'dy']
        for dim in range(2):
            ax.plot(steps, goal[:, dim], ':', color=colors[dim], alpha=0.9,
                    linewidth=1.5, label=f'decoded {goal_labels[dim]}')
    else:
        ax.text(0.5, 0.5, 'Goal not available', ha='center', va='center',
                transform=ax.transAxes)
    ax.text(0.02, 0.97, '(a)', transform=ax.transAxes, fontsize=16,
            fontweight='bold', va='top', ha='left')
    ax.set_xlabel('Time step')
    ax.set_ylabel('Goal position (m)')
    ax.legend(fontsize=11, ncol=2, loc='upper right')
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='k', linestyle='--', alpha=0.2)

    # ── (b) Decoded reward ────────────────────────────────────────────────────
    ax = axes[1, 0]
    if has_reward:
        rewards = np.array(rollout_policy['rewards'])[:, 0]
        ax.plot(steps, rewards, ':', color='C0', linewidth=1.5, label='predicted reward ')
    else:
        ax.text(0.5, 0.5, 'Reward not available', ha='center', va='center',
                transform=ax.transAxes)
    ax.text(0.02, 0.97, '(b)', transform=ax.transAxes, fontsize=16,
            fontweight='bold', va='top', ha='left')
    ax.set_xlabel('Time step')
    ax.set_ylabel('Reward')
    ax.legend(fontsize=11, ncol=1, loc='upper right')
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='k', linestyle='--', alpha=0.2)

    plt.tight_layout()
    return fig


def plot_gt_comparison(rollout_policy, run_label, obs_batch_ref=None):
    """
    Two-panel figure comparing GT episode data vs imagined rollout:
    (a) Goal comparison: GT episode vs imagined rollout
    (b) Reward comparison: GT episode vs imagined rollout
    """
    has_goal   = 'goal'    in rollout_policy
    has_reward = 'rewards' in rollout_policy

    actions = np.array(rollout_policy['actions'])[:, 0, :]
    H       = len(actions)
    steps   = np.arange(H)

    colors = list(plt.cm.tab10.colors)

    fig, axes = plt.subplots(2, 1, figsize=(10, 4 * 2), squeeze=False)

    # ── (a) Goal comparison: GT episode vs imagined rollout ───────────────────
    ax = axes[0, 0]
    if has_goal and obs_batch_ref is not None and 'goal' in obs_batch_ref:
        goal_imag = np.array(rollout_policy['goal'])          # (H, 2)
        goal_real = obs_batch_ref['goal'][0]                  # (T, 2)
        real_steps = np.arange(min(len(goal_real), H))
        goal_labels = ['dx', 'dy']
        for dim in range(2):
            ax.plot(real_steps, goal_real[:H, dim], '-', color=colors[dim],
                    linewidth=1.5, alpha=0.85, label=f'GT {goal_labels[dim]}')

            # Shift the imagined trace by one step so it starts on the same
            # initial observation value as the GT trace.
            imag_shifted = np.concatenate([goal_real[:1, dim], goal_imag[:-1, dim]])
            ax.plot(steps, imag_shifted, ':', color=colors[dim],
                    linewidth=1.5, alpha=0.85, label=f'imagined {goal_labels[dim]} ')
    elif has_goal:
        goal_imag = np.array(rollout_policy['goal'])
        goal_labels = ['dx', 'dy']
        for dim in range(2):
            ax.plot(steps, goal_imag[:, dim], ':', color=colors[dim], alpha=0.9,
                    linewidth=1.5, label=f'imagined {goal_labels[dim]} ')
        ax.text(0.5, 0.02, 'No GT episode data provided for comparison',
                ha='center', va='bottom', transform=ax.transAxes, fontsize=8, color='gray')
    else:
        ax.text(0.5, 0.5, 'Goal not available', ha='center', va='center',
                transform=ax.transAxes)
    ax.text(0.02, 0.97, '(a)', transform=ax.transAxes, fontsize=16,
            fontweight='bold', va='top', ha='left')
    ax.set_xlabel('Time step')
    ax.set_ylabel('Goal position (m)')
    ax.legend(fontsize=11, ncol=2, loc='upper right')
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='k', linestyle='--', alpha=0.2)

    # ── (b) Reward comparison: GT episode vs imagined rollout ─────────────────
    ax = axes[1, 0]
    if has_reward and obs_batch_ref is not None and 'reward' in obs_batch_ref:
        rewards_imag = np.array(rollout_policy['rewards'])[:, 0]   # (H,)
        rewards_real = obs_batch_ref['reward'][0]                   # (T,)
        real_steps   = np.arange(min(len(rewards_real), H))
        ax.plot(real_steps, rewards_real[:H], '-', color='C0',
                linewidth=1.5, alpha=0.85, label='GT reward')
        ax.plot(steps, rewards_imag, ':', color='C0',
                linewidth=1.5, alpha=0.85, label='predicted reward ')
    elif has_reward:
        rewards_imag = np.array(rollout_policy['rewards'])[:, 0]
        ax.plot(steps, rewards_imag, ':', color='C0', linewidth=1.5, label='predicted reward ')
        ax.text(0.5, 0.02, 'No GT episode data provided for comparison',
                ha='center', va='bottom', transform=ax.transAxes, fontsize=8, color='gray')
    else:
        ax.text(0.5, 0.5, 'Reward not available', ha='center', va='center',
                transform=ax.transAxes)
    ax.text(0.02, 0.97, '(b)', transform=ax.transAxes, fontsize=16,
            fontweight='bold', va='top', ha='left')
    ax.set_xlabel('Time step')
    ax.set_ylabel('Reward')
    ax.legend(fontsize=11, ncol=1, loc='upper right')
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='k', linestyle='--', alpha=0.2)

    plt.tight_layout()
    return fig

## Execute Rollouts

Run the cell below to evaluate the policy rollout for each selected run.

**Policy rollout** (30 timesteps ≈ 10 seconds at 3 fps): Starting from an encoded real observation frame, the trained actor-critic drives the imagination forward — the actor selects each action, the RSSM steps the latent state, and the critic scores it. Every dreamed latent state is decoded back to an image so you can watch the robot's imagined trajectory, predicted velocity, perceived goal direction, and estimated value/reward at each step.

Training used a horizon of 15, so evaluating at 30 allows room for longer-horizon behavior without excessive extrapolation.

In [ ]:

def compute_latent_drift(wm, agent_state, obs_batch, actions_batch, horizon=None):
    """
    Compute L2 distance between consecutive latent states during rollout.
    Measures how much the RSSM's latent state drifts over the imagination horizon.
    """
    if horizon is None:
        horizon = min(obs_batch['is_first'].shape[1] - 1, 50)

    def drift_analysis():
        first_obs = {k: v[:, 0:1] for k, v in obs_batch.items()}
        first_action = jnp.zeros((obs_batch['is_first'].shape[0], 1, 3), dtype=jnp.float32)
        embed_first = wm.encoder(first_obs)
        post_first, _ = wm.rssm.observe(embed_first, first_action, first_obs['is_first'])
        latent = {k: v[:, 0] for k, v in post_first.items()}

        deter_drift = [0.0]
        stoch_drift = [0.0]

        for t in range(horizon):
            # Random action
            action = jnp.tanh(jax.random.normal(nj.rng(), shape=(latent['deter'].shape[0], 1, 3)))
            next_latent = wm.rssm.imagine(action, latent)
            next_latent_t0 = {k: v[:, 0] for k, v in next_latent.items()}

            # Compute L2 drift
            deter_dist = jnp.linalg.norm(next_latent_t0['deter'] - latent['deter'])
            stoch_dist = jnp.linalg.norm(next_latent_t0['stoch'] - latent['stoch'])

            deter_drift.append(deter_dist)
            stoch_drift.append(stoch_dist)
            latent = next_latent_t0

        return {
            'deter': jnp.array(deter_drift),
            'stoch': jnp.array(stoch_drift),
        }

    rng_key = jax.random.PRNGKey(42)
    drift_result, _ = nj.pure(drift_analysis)(agent_state, rng_key)
    return {
        'deter': np.array(drift_result['deter']),
        'stoch': np.array(drift_result['stoch']),
    }


def compute_reconstruction_error(wm, agent_state, obs_batch, actions_batch, obs_keys, horizon=None):
    """
    Compute MSE between decoded world-model predictions and real observations.
    """
    if horizon is None:
        horizon = min(obs_batch['is_first'].shape[1], 50)

    obs_keys_to_eval = [k for k in obs_keys if k in ['velocity', 'orientation', 'goal', 'position']]

    def recon_error_analysis():
        first_obs = {k: v[:, 0:1] for k, v in obs_batch.items()}
        first_action = jnp.zeros((obs_batch['is_first'].shape[0], 1, 3), dtype=jnp.float32)
        embed_first = wm.encoder(first_obs)
        post_first, _ = wm.rssm.observe(embed_first, first_action, first_obs['is_first'])
        latent = {k: v[:, 0] for k, v in post_first.items()}

        errors = {k: [] for k in obs_keys_to_eval}

        for t in range(horizon):
            # Decode latent state
            latent_expanded = {k: jnp.expand_dims(v, 1) for k, v in latent.items()}
            recons_dists = wm.heads['decoder'](latent_expanded)

            # Compute MSE against real observation
            for key in obs_keys_to_eval:
                if key in recons_dists:
                    pred = recons_dists[key].mean()[:, 0]  # (B, D)
                    real = obs_batch[key][:, t]  # (B, D)
                    mse = jnp.mean((pred - real) ** 2)
                    errors[key].append(mse)

            # Step forward with random action
            action = jnp.tanh(jax.random.normal(nj.rng(), shape=(latent['deter'].shape[0], 1, 3)))
            next_latent = wm.rssm.imagine(action, latent)
            latent = {k: v[:, 0] for k, v in next_latent.items()}

        return {k: jnp.array(v) for k, v in errors.items()}

    rng_key = jax.random.PRNGKey(42)
    error_result, _ = nj.pure(recon_error_analysis)(agent_state, rng_key)
    return {k: np.array(v) for k, v in error_result.items()}


def rollout_stability_metrics(rollout_policy):
    """
    Compute stability metrics from a policy rollout:
    - action_smoothness: mean absolute difference between consecutive actions
    - mean_policy_entropy: mean entropy of the policy distribution
    - action_saturation: fraction of timesteps where actions are near ±1
    """
    actions = np.array(rollout_policy['actions'])[:, 0, :]  # (H, 3)
    entropies = np.array(rollout_policy.get('entropies', []))
    if entropies.ndim == 2:
        entropies = entropies[:, 0]

    # Action smoothness: mean absolute difference between consecutive actions
    action_diffs = np.abs(np.diff(actions, axis=0))
    action_smoothness = float(np.mean(action_diffs))

    # Mean policy entropy
    mean_entropy = float(np.mean(entropies)) if len(entropies) > 0 else 0.0

    # Action saturation: fraction of actions near ±1
    saturation_mask = np.abs(actions) > 0.95
    action_saturation = float(np.mean(saturation_mask))

    return {
        'action_smoothness': action_smoothness,
        'mean_policy_entropy': mean_entropy,
        'action_saturation': action_saturation,
    }


print("✓ Helper analysis functions defined: compute_latent_drift, compute_reconstruction_error, rollout_stability_metrics")


In [ ]:

print("\n" + "="*80)
print("EXECUTING POLICY EVALUATIONS")
print("="*80 + "\n")

ROLLOUT_START_STEP = 0

if _raw_data is None:
    print("⚠ Skipping: no episode data loaded")
else:
    rollout_results = {}

    for run_idx in selected_runs:
        agent, agent_state = run_agents[run_idx]
        wm        = run_wms[run_idx]
        obs_keys  = run_obs_keys[run_idx]
        run_label = f"Run {run_idx}: {available_runs[run_idx].name}"

        # Build obs_batch filtered to this run's obs_keys
        obs_batch_run  = make_obs_batch(obs_keys)
        actions_batch  = actions_batch_real

        print(f"\nEvaluating: {run_label}")
        print(f"  obs_keys: {obs_keys}")
        print(f"  Start frame: {ROLLOUT_START_STEP}")
        print("-" * 80)

        try:
            print("  Running policy rollout...")
            policy_rollout, _ = rollout_with_policy(
                agent, wm, agent_state, obs_batch_run, actions_batch, obs_keys,
                horizon=38, start_step=ROLLOUT_START_STEP,
            )
            print(f"    ✓ Action mean magnitude: {np.mean(np.abs(policy_rollout['actions'])):.4f}")
            print(f"    ✓ Value mean:   {np.mean(policy_rollout['values']):.4f}")
            print(f"    ✓ Reward mean:  {np.mean(policy_rollout['rewards']):.4f}")
            print(f"    ✓ Entropy mean: {np.mean(policy_rollout['entropies']):.4f}")
            if 'is_terminal' in policy_rollout:
                print(f"    ✓ is_terminal max probability: {np.max(policy_rollout['is_terminal']):.4f}")
            if 'cont' in policy_rollout:
                cont_arr = np.array(policy_rollout['cont'])
                if cont_arr.ndim == 2: cont_arr = cont_arr[:, 0]
                print(f"    ✓ cont (episode continuation): min={cont_arr.min():.3f}  mean={cont_arr.mean():.3f}  (1-cont = P(terminal))")
            for key in obs_keys:
                if key in policy_rollout and key not in ('reward',):
                    print(f"    ✓ Decoded {key}: {policy_rollout[key].shape}")

            v_flat = np.array(policy_rollout['values']).flatten()
            r_flat = np.array(policy_rollout['rewards']).flatten()
            if np.std(v_flat) > 1e-6 and np.std(r_flat) > 1e-6:
                print(f"    ✓ Value-reward corr: {np.corrcoef(v_flat, r_flat)[0,1]:.4f}")

            print("  Running latent drift analysis...")
            drift_dict = compute_latent_drift(wm, agent_state, obs_batch_run, actions_batch)
            print(f"    ✓ Final deter drift: {drift_dict['deter'][-1]:.4f}")
            print(f"    ✓ Final stoch drift: {drift_dict['stoch'][-1]:.4f}")

            print("  Running reconstruction error analysis...")
            recon_errors = compute_reconstruction_error(wm, agent_state, obs_batch_run, actions_batch, obs_keys)
            for mod, err in recon_errors.items():
                print(f"    ✓ {mod} MSE: {np.mean(err):.6f}")

            stability = rollout_stability_metrics(policy_rollout)
            print(f"  Stability: smoothness={stability['action_smoothness']:.4f}  "
                  f"entropy={stability['mean_policy_entropy']:.4f}  "
                  f"saturation={stability['action_saturation']:.4f}")

            rollout_results[run_idx] = {
                'policy':       policy_rollout,
                'label':        run_label,
                'latent_drift': drift_dict,
                'recon_errors': recon_errors,
                'stability':    stability,
                'obs_keys':     obs_keys,
                'obs_batch':    obs_batch_run,
            }

        except Exception as e:
            import traceback
            print(f"    ✗ Error: {e}")
            traceback.print_exc()

    print(f"\n{'='*80}")
    print(f"✓ Completed {len(rollout_results)}/{len(selected_runs)} evaluations")


In [ ]:
from pathlib import Path

if rollout_results:
    print("\n" + "="*80)
    print("DETAILED ROLLOUT VISUALIZATIONS")
    print("="*80 + "\n")

    for run_idx, results in rollout_results.items():
        print(f"\nGenerating visualizations for: {results['label']}")
        print("-" * 80)

        # Get the run directory for this run
        run_dir = available_runs[run_idx]
        save_dir = run_dir / 'agentcritic_results'
        save_dir.mkdir(parents=True, exist_ok=True)

        saved_files = []

        # Save GIF
        gif_path = save_dir / f'policy_dream_run{run_idx}.gif'
        create_policy_state_gif(results['policy'], results['label'],
                                obs_batch_ref=results['obs_batch'],
                                output_path=str(gif_path), fps=3)
        saved_files.append(('Policy Dream GIF', gif_path))

        # Save action sequence plot
        fig = plot_action_sequences(results['policy'], results['label'])
        act_path = save_dir / f'action_sequences_run{run_idx}.png'
        fig.savefig(act_path, dpi=100, bbox_inches='tight')
        saved_files.append(('Action Sequences', act_path))
        plt.show()
        plt.close(fig)

        # Save value predictions plot
        fig = plot_value_predictions(results['policy'], results['label'])
        val_path = save_dir / f'value_predictions_run{run_idx}.png'
        fig.savefig(val_path, dpi=100, bbox_inches='tight')
        saved_files.append(('Value Predictions', val_path))
        plt.show()
        plt.close(fig)

        # Save policy entropy plot
        fig = plot_policy_entropy(results['policy'], results['label'])
        if fig is not None:
            ent_path = save_dir / f'policy_entropy_run{run_idx}.png'
            fig.savefig(ent_path, dpi=100, bbox_inches='tight')
            saved_files.append(('Policy Entropy', ent_path))
            plt.show()
            plt.close(fig)

        # Save imagined goal + reward plot (a/b)
        fig = plot_command_state_alignment(results['policy'], results['label'])
        cmd_path = save_dir / f'imagined_goal_reward_run{run_idx}.png'
        fig.savefig(cmd_path, dpi=100, bbox_inches='tight')
        saved_files.append(('Imagined Goal & Reward', cmd_path))
        plt.show()
        plt.close(fig)

        # Save GT vs imagined comparison plot (a/b)
        fig = plot_gt_comparison(results['policy'], results['label'], obs_batch_ref=results['obs_batch'])
        gt_path = save_dir / f'gt_comparison_run{run_idx}.png'
        fig.savefig(gt_path, dpi=100, bbox_inches='tight')
        saved_files.append(('GT vs Imagined Comparison', gt_path))
        plt.show()
        plt.close(fig)

        # Save latent drift plot
        fig = plot_latent_drift(results['latent_drift'], results['label'])
        drift_path = save_dir / f'latent_drift_run{run_idx}.png'
        fig.savefig(drift_path, dpi=100, bbox_inches='tight')
        saved_files.append(('Latent Drift', drift_path))
        plt.show()
        plt.close(fig)

        # Save reconstruction error plot
        fig = plot_reconstruction_error(results['recon_errors'], results['label'])
        if fig is not None:
            recon_path = save_dir / f'reconstruction_error_run{run_idx}.png'
            fig.savefig(recon_path, dpi=100, bbox_inches='tight')
            saved_files.append(('Reconstruction Error', recon_path))
            plt.show()
            plt.close(fig)

        # Save cont prediction plot (episode termination via cont head)
        if 'cont' in results['policy']:
            fig = plot_cont_prediction(results['policy'], results['label'], config=run_configs.get(run_idx))
            if fig is not None:
                cont_path = save_dir / f'cont_prediction_run{run_idx}.png'
                fig.savefig(cont_path, dpi=100, bbox_inches='tight')
                saved_files.append(('Continuation Prediction', cont_path))
                plt.show()
                plt.close(fig)

        # Print summary of saved files with clickable links
        print(f"\n✓ Saved {len(saved_files)} files to: {save_dir}")
        for label, fpath in saved_files:
            print(f"  • {label:30s} → file://{fpath}")

    print(f"\n{'='*80}")
    print(f"✓ Completed visualizations for {len(rollout_results)} run(s)")
else:
    print("⚠ No rollout results available. Run the previous cell first.")
